# exp004 Pseudo-Labeling Pipeline (ONNX + soundfile)

GPU ノートブック。ONNX Perch v2 で全 train_soundscapes の埋め込みを計算し、
疑似ラベリングによる ProtoSSM + MLP の学習を行う。

出力:
- `all_perch_embeddings.npz` — 全 train_soundscapes の Perch 埋め込みキャッシュ
- `pseudo_label_weights.pt` — 疑似ラベリング済み ProtoSSM 重み
- `pseudo_label_probes.pkl` — 疑似ラベリング済み MLP Probe


In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'onnxruntime-gpu', 'huggingface_hub'])

In [ ]:
# Cell 2 — Imports
import os, gc, json, re, time, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm

import onnxruntime as ort

warnings.filterwarnings("ignore")

DEVICE = "cpu"  # ProtoSSM is lightweight, runs on CPU
print("onnxruntime version:", ort.__version__)
print("CUDA available:", "CUDAExecutionProvider" in ort.get_available_providers())

In [ ]:
# Cell 3 — Config
BASE = Path("/kaggle/input/birdclef-2026")
if not BASE.exists():
    BASE = Path("/kaggle/input/competitions/birdclef-2026")

SR = 32000
DURATION = 60
FILE_SAMPLES = SR * DURATION
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC  # 160000
N_WINDOWS = DURATION // WINDOW_SEC  # 12

CFG = {
    "mode": "train",
    "verbose": True,
    "batch_files": 4,  # 4 files * 12 windows = 48 segments/batch (T4 16GB safe)
    "proxy_reduce": "max",
    "n_splits": 5,
    # ProtoSSM
    "proto_ssm": {
        "d_model": 128,
        "d_state": 16,
        "n_ssm_layers": 2,
        "dropout": 0.1,
    },
    "proto_ssm_train": {
        "n_epochs": 80,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "patience": 15,
        "bce_weight": 1.0,
        "distill_weight": 0.3,
        "family_weight": 0.1,
        "val_ratio": 0.15,
        "pos_weight_cap": 50.0,
    },
    # MLP
    "mlp_params": {
        "hidden_layer_sizes": (128,),
        "activation": "relu",
        "max_iter": 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "random_state": 42,
    },
    "frozen_best_probe": {
        "alpha": 0.6,
        "min_pos": 2,
    },
    # Fusion
    "best_fusion": {
        "lambda_event": 0.4,
        "lambda_texture": 1.0,
        "lambda_proxy_texture": 0.8,
        "smooth_texture": 0.35,
        "smooth_event": 0.15,
    },
    # PCA
    "pca_dim": 64,
    # Pseudo-labeling
    "pseudo_rounds": 3,
    "pseudo_power": [1.0, 1.54, 1.82],  # BirdCLEF 2025 1st: 1, 1/0.65, 1/0.55
    "pseudo_threshold": 0.5,  # minimum probability to include as pseudo-label
}

BEST = CFG["best_fusion"]

# ONNX model path (will be downloaded)
ONNX_MODEL_PATH = None

# Output directory
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

print("BASE:", BASE)
print("Config ready.")

In [ ]:
# Cell 4 — Download ONNX Perch v2 model
from huggingface_hub import hf_hub_download

print("Downloading Perch v2 ONNX model from HuggingFace...")
t0 = time.time()
ONNX_MODEL_PATH = hf_hub_download(
    repo_id="justinchuby/Perch-onnx",
    filename="perch_v2.onnx",
)
print(f"Downloaded in {time.time()-t0:.1f}s: {ONNX_MODEL_PATH}")

# Load ONNX session with GPU
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
onnx_session = ort.InferenceSession(ONNX_MODEL_PATH, providers=providers)
print("Active providers:", onnx_session.get_providers())

# Verify input/output
for inp in onnx_session.get_inputs():
    print(f"Input: {inp.name}, shape={inp.shape}")
for out in onnx_session.get_outputs():
    print(f"Output: {out.name}, shape={out.shape}")

In [ ]:
# Cell 5 — Load taxonomy and labels
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sorted(taxonomy["primary_label"].tolist())
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

# Parse soundscape labels into multi-hot matrix
ss_files = sorted(soundscape_labels["filename"].unique().tolist())
print(f"Labeled soundscape files: {len(ss_files)}")

# Build label matrix for labeled files
all_ss_dir = BASE / "train_soundscapes"
all_ss_files = sorted([f.name for f in all_ss_dir.glob("*.ogg")])
print(f"Total soundscape files: {len(all_ss_files)}")
print(f"Unlabeled soundscape files: {len(all_ss_files) - len(ss_files)}")

# Build row-level labels for labeled soundscapes
def build_label_rows(label_df, primary_labels, label_to_idx):
    rows = []
    for _, r in label_df.iterrows():
        fn = r["filename"]
        start_sec = int(pd.Timedelta(r["start"]).total_seconds())
        end_sec = int(pd.Timedelta(r["end"]).total_seconds())
        labels = str(r["primary_label"]).split(";")
        row_id = f"{Path(fn).stem}_{end_sec}"
        y = np.zeros(len(primary_labels), dtype=np.float32)
        for lbl in labels:
            lbl = lbl.strip()
            if lbl in label_to_idx:
                y[label_to_idx[lbl]] = 1.0
        rows.append({"row_id": row_id, "filename": fn, "end_sec": end_sec, "y": y})
    return rows

label_rows = build_label_rows(soundscape_labels, PRIMARY_LABELS, label_to_idx)
print(f"Label rows: {len(label_rows)}")

In [ ]:
# Cell 6 — Perch label mapping and genus proxies
# Load Perch labels from the ONNX model's associated labels
# We need labels.csv - try to find it from Kaggle model input or download
import csv

# Try to find labels.csv from Perch model input
LABELS_CSV = None
for candidate in [
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv"),
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/1/assets/labels.csv"),
    Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2/assets/labels.csv"),
]:
    if candidate.exists():
        LABELS_CSV = candidate
        break

if LABELS_CSV is None:
    # Download from HuggingFace
    LABELS_CSV = hf_hub_download(
        repo_id="google/bird-vocalization-classifier",
        filename="assets/labels.csv",
        revision="main",
    )
    # If that fails, we'll handle it
    if LABELS_CSV is None:
        raise FileNotFoundError("Cannot find Perch labels.csv")

print(f"Perch labels: {LABELS_CSV}")

bc_labels = (
    pd.read_csv(LABELS_CSV)
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

taxonomy_m = taxonomy.copy()
taxonomy_m["scientific_name_lookup"] = taxonomy_m["scientific_name"]
bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy_m.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}

# Genus proxies for unmapped species
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "genus": genus,
            "bc_indices": hits["bc_index"].astype(int).tolist(),
        }

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)

selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

# Build ACTIVE_CLASSES from labeled data
_active_set = set()
for r in label_rows:
    _active_set.update(np.where(r["y"] > 0)[0])
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in sorted(_active_set)]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]

idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32
)

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Mapped: {MAPPED_MASK.sum()}/{N_CLASSES}, Proxies: {len(SELECTED_PROXY_TARGETS)}")
print(f"Active classes: {len(ACTIVE_CLASSES)} (texture: {len(idx_active_texture)}, event: {len(idx_active_event)})")

In [ ]:
# Cell 7 — ONNX Perch inference (soundfile + batched)

def parse_soundscape_filename(fname):
    parts = Path(fname).stem.split("_")
    site = [p for p in parts if p.startswith("S")][0] if any(p.startswith("S") for p in parts) else "UNK"
    # Extract hour from time part (last element, HHMMSS format)
    time_str = parts[-1] if len(parts) >= 6 else "000000"
    hour_utc = int(time_str[:2]) if len(time_str) >= 2 else 0
    return {"site": site, "hour_utc": hour_utc}

def read_soundscape_sf(path):
    """Read 60s soundscape using soundfile (fast, no resampling)."""
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_onnx(paths, batch_files=4, verbose=True):
    """Compute Perch embeddings + logits for soundscape files using ONNX.
    
    Note: batch_files=4 means 4 files * 12 windows = 48 segments per batch.
    Kaggle T4 GPU (16GB) can handle ~48 segments safely.
    """
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files,
                       desc="ONNX Perch")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_sf(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        # ONNX inference
        outputs = onnx_session.run(None, {"inputs": x})
        output_names = [o.name for o in onnx_session.get_outputs()]
        out_dict = dict(zip(output_names, outputs))

        logits = out_dict["label"].astype(np.float32)
        emb = out_dict["embedding"].astype(np.float32)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        # Genus proxies
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            scores[batch_row_start:write_row, pos] = sub.max(axis=1).astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })

    return meta_df, scores, embeddings

print("ONNX Perch inference function ready. (batch_files=4, 48 segments/batch)")

In [ ]:
# Cell 8 — Compute Perch embeddings for ALL train_soundscapes

CACHE_PATH = OUT_DIR / "all_perch_embeddings.npz"
CACHE_META_PATH = OUT_DIR / "all_perch_meta.parquet"

if CACHE_PATH.exists() and CACHE_META_PATH.exists():
    print("Loading cached embeddings...")
    arr = np.load(CACHE_PATH)
    all_scores = arr["scores"]
    all_embeddings = arr["embeddings"]
    all_meta = pd.read_parquet(CACHE_META_PATH)
else:
    all_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))
    print(f"Computing embeddings for {len(all_paths)} files...")

    t0 = time.time()
    all_meta, all_scores, all_embeddings = infer_perch_onnx(
        all_paths, batch_files=CFG["batch_files"], verbose=True
    )
    elapsed = time.time() - t0
    print(f"Done in {elapsed/60:.1f} min ({elapsed/3600:.2f} hr)")

    # Save cache
    np.savez_compressed(CACHE_PATH, scores=all_scores, embeddings=all_embeddings)
    all_meta.to_parquet(CACHE_META_PATH, index=False)
    print(f"Saved cache: {CACHE_PATH} ({CACHE_PATH.stat().st_size/1e6:.1f} MB)")

print(f"All meta: {all_meta.shape}")
print(f"All scores: {all_scores.shape}")
print(f"All embeddings: {all_embeddings.shape}")

In [ ]:
# Cell 9 — Split labeled vs unlabeled data

labeled_filenames = set(ss_files)
is_labeled = all_meta["filename"].isin(labeled_filenames).values

# Labeled data
meta_labeled = all_meta[is_labeled].reset_index(drop=True)
scores_labeled = all_scores[is_labeled]
emb_labeled = all_embeddings[is_labeled]

# Build Y_labeled (ground truth labels)
label_row_map = {}
for row in label_rows:
    label_row_map[row["row_id"]] = row["y"]

Y_labeled = np.zeros((len(meta_labeled), N_CLASSES), dtype=np.float32)
for i, row_id in enumerate(meta_labeled["row_id"]):
    if row_id in label_row_map:
        Y_labeled[i] = label_row_map[row_id]

# Unlabeled data
meta_unlabeled = all_meta[~is_labeled].reset_index(drop=True)
scores_unlabeled = all_scores[~is_labeled]
emb_unlabeled = all_embeddings[~is_labeled]

print(f"Labeled: {len(meta_labeled)} windows ({len(labeled_filenames)} files)")
print(f"Unlabeled: {len(meta_unlabeled)} windows ({meta_unlabeled['filename'].nunique()} files)")
print(f"Active classes in labeled: {int((Y_labeled.sum(axis=0) > 0).sum())}")

In [ ]:
# Cell 4 — Metrics and helper utilities
def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")

def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()

    s = scores.copy()
    assert len(s) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
    view = s.reshape(-1, N_WINDOWS, s.shape[1])

    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)

    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s

def smooth_events_fixed12(scores, cols, alpha=0.15):
    """Soft max-pool context for event birds (Aves).
    Uses local_max instead of average neighbor, preserving transient call detection."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s

def seq_features_1d(v):
    """
    v: shape (n_rows,), ordered as full-file blocks of 12 windows
    Extended: tambah std_v untuk capture variance temporal dalam file
    """
    assert len(v) % N_WINDOWS == 0, "Expected full-file blocks of 12 windows"
    x = v.reshape(-1, N_WINDOWS)

    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v  = np.repeat(x.max(axis=1),  N_WINDOWS)
    std_v  = np.repeat(x.std(axis=1),  N_WINDOWS)

    return prev_v, next_v, mean_v, max_v, std_v

In [ ]:
# Cell 7 — Fold-safe metadata prior tables
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)

    global_p = Y_prior.mean(axis=0).astype(np.float32)

    # Site
    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)

    for s in site_keys:
        i = site_to_i[s]
        mask = prior_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_prior[mask].mean(axis=0)

    # Hour
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)

    for h in hour_keys:
        i = hour_to_i[h]
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_prior[mask].mean(axis=0)

    # Site-hour
    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []

    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_n": site_n,
        "site_p": site_p,
        "hour_to_i": hour_to_i,
        "hour_n": hour_n,
        "hour_p": hour_p,
        "sh_to_i": sh_to_i,
        "sh_n": sh_n,
        "sh_p": sh_p,
    }

def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter(
        (tables["site_to_i"].get(str(s), -1) for s in sites),
        dtype=np.int32,
        count=n
    )
    hour_idx = np.fromiter(
        (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        dtype=np.int32,
        count=n
    )
    sh_idx = np.fromiter(
        (tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)),
        dtype=np.int32,
        count=n
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)

def fuse_scores_with_tables(base_scores, sites, hours, tables,
                            lambda_event=BEST["lambda_event"],
                            lambda_texture=BEST["lambda_texture"],
                            lambda_proxy_texture=BEST["lambda_proxy_texture"],
                            smooth_texture=BEST["smooth_texture"],
                            smooth_event=BEST["smooth_event"]):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    # mapped active
    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]

    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]

    # selected frog proxies
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]

    # prior-only active unmapped
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = lambda_event * prior[:, idx_selected_prioronly_active_event]

    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = lambda_texture * prior[:, idx_selected_prioronly_active_texture]

    # inactive unmapped
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=smooth_event)
    return scores.astype(np.float32, copy=False), prior

In [ ]:
# Cell 9 — Classwise embedding-probe helpers
def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """
    emb_proj: (n, d)
    raw_col, prior_col, base_col: (n,)
    returns: (n, d + 13)

    Fitur: embedding + 7 sequential + 3 interaction + std + 3 diff
    """
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)

    # Diff features: posisi window relatif terhadap konteks file
    diff_mean = base_col - mean_base   # apakah window ini lebih tinggi dari rata2 file?
    diff_prev = base_col - prev_base   # onset: naik dari window sebelumnya?
    diff_next = base_col - next_base   # offset: turun ke window berikutnya?

    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],             # variance temporal dalam file
        diff_mean[:, None],            # deviasi dari mean file
        diff_prev[:, None],            # deteksi onset
        diff_next[:, None],            # deteksi offset
        # interaction terms
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)

    return feats.astype(np.float32, copy=False)

def run_oof_embedding_probe(
    scores_raw,
    emb,
    meta_df,
    y_true,
    pca_dim=64,
    min_pos=8,
    C=0.25,
    alpha=0.5,
):
    groups = meta_df["filename"].to_numpy()
    gkf = GroupKFold(n_splits=5)

    oof_base_local = np.zeros_like(scores_raw, dtype=np.float32)
    oof_final = np.zeros_like(scores_raw, dtype=np.float32)

    modeled_counts = np.zeros(scores_raw.shape[1], dtype=np.int32)

    split_list = list(gkf.split(scores_raw, groups=groups))

    for fold, (tr_idx, va_idx) in enumerate(tqdm(split_list, desc="Embedding-probe folds", disable=not CFG["verbose"]), 1):
    # for fold, (tr_idx, va_idx) in enumerate(tqdm(split_list, desc="Embedding-probe folds"), 1):
        tr_idx = np.sort(tr_idx)
        va_idx = np.sort(va_idx)

        val_files = set(meta_df.iloc[va_idx]["filename"].tolist())

        # Fold-safe priors
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
        Y_prior_fold = Y_SC[prior_mask]
        tables = fit_prior_tables(prior_df_fold, Y_prior_fold)

        base_tr, prior_tr = fuse_scores_with_tables(
            scores_raw[tr_idx],
            sites=meta_df.iloc[tr_idx]["site"].to_numpy(),
            hours=meta_df.iloc[tr_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )
        base_va, prior_va = fuse_scores_with_tables(
            scores_raw[va_idx],
            sites=meta_df.iloc[va_idx]["site"].to_numpy(),
            hours=meta_df.iloc[va_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )

        oof_base_local[va_idx] = base_va
        oof_final[va_idx] = base_va

        # Embedding preprocessing on train fold only
        scaler = StandardScaler()
        emb_tr_s = scaler.fit_transform(emb[tr_idx])
        emb_va_s = scaler.transform(emb[va_idx])

        n_comp = min(pca_dim, emb_tr_s.shape[0] - 1, emb_tr_s.shape[1])
        pca = PCA(n_components=n_comp)
        Z_tr = pca.fit_transform(emb_tr_s).astype(np.float32)
        Z_va = pca.transform(emb_va_s).astype(np.float32)

        class_iterator = np.where(y_true[tr_idx].sum(axis=0) >= min_pos)[0].tolist()

        for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False, disable=not CFG["verbose"]):
        # for cls_idx in tqdm(class_iterator, desc=f"Fold {fold} classes", leave=False):
            y_tr = y_true[tr_idx, cls_idx]

            if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                continue

            X_tr_cls = build_class_features(
                Z_tr,
                raw_col=scores_raw[tr_idx, cls_idx],
                prior_col=prior_tr[:, cls_idx],
                base_col=base_tr[:, cls_idx],
            )
            X_va_cls = build_class_features(
                Z_va,
                raw_col=scores_raw[va_idx, cls_idx],
                prior_col=prior_va[:, cls_idx],
                base_col=base_va[:, cls_idx],
            )

            # Pilih backend probe: mlp | lgbm | logreg
            backend = CFG.get("probe_backend", "mlp")
            n_pos = int(y_tr.sum())
            n_neg = len(y_tr) - n_pos

            if backend == "mlp":
                # MLPClassifier tidak support sample_weight
                # Gunakan oversampling: duplikasi positif agar balance
                if n_pos > 0 and n_neg > n_pos:
                    repeat = max(1, n_neg // n_pos)
                    pos_idx = np.where(y_tr == 1)[0]
                    X_bal = np.vstack([X_tr_cls, np.tile(X_tr_cls[pos_idx], (repeat, 1))])
                    y_bal = np.concatenate([y_tr, np.ones(len(pos_idx) * repeat, dtype=y_tr.dtype)])
                else:
                    X_bal, y_bal = X_tr_cls, y_tr
                clf = MLPClassifier(**CFG["mlp_params"])
                clf.fit(X_bal, y_bal)
                pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
            elif backend == "lgbm" and _LGBM_AVAILABLE:
                scale_pos = max(1.0, n_neg / max(n_pos, 1))
                clf = LGBMClassifier(
                    **CFG["lgbm_params"],
                    scale_pos_weight=scale_pos,
                )
                clf.fit(X_tr_cls, y_tr)
                pred_va = clf.predict_proba(X_va_cls)[:, 1].astype(np.float32)
                pred_va = np.log(pred_va + 1e-7) - np.log(1 - pred_va + 1e-7)
            else:
                clf = LogisticRegression(
                    C=C, max_iter=400, solver="liblinear",
                    class_weight="balanced",
                )
                clf.fit(X_tr_cls, y_tr)
                pred_va = clf.decision_function(X_va_cls).astype(np.float32)

            oof_final[va_idx, cls_idx] = (
                (1.0 - alpha) * base_va[:, cls_idx] +
                alpha * pred_va
            )

            modeled_counts[cls_idx] += 1

    score_base = macro_auc_skip_empty(y_true, oof_base_local)
    score_final = macro_auc_skip_empty(y_true, oof_final)

    return {
        "oof_base": oof_base_local,
        "oof_final": oof_final,
        "modeled_counts": modeled_counts,
        "score_base": score_base,
        "score_final": score_final,
    }

In [ ]:
# ProtoSSM — Prototypical State Space Model

class SelectiveSSM(nn.Module):
    """Simplified Mamba-style selective state space model.
    
    Input-dependent (selective) discretization of continuous-time SSM:
        dx/dt = Ax + Bu,  y = Cx + Du
    where A, B, C are functions of the input (selectivity).
    
    For T=12 bioacoustic windows, the sequential scan is efficient on CPU.
    """
    
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # Input projection: x -> (x_ssm, z_gate)
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        
        # Causal conv1d for local context before SSM
        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv,
            padding=d_conv - 1, groups=d_model
        )
        
        # SSM parameters
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        
        # A initialized as structured matrix (HiPPO-inspired)
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        
        # D is the skip connection
        self.D = nn.Parameter(torch.ones(d_model))
        
        # B and C projections — input-dependent = selective
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x):
        """x: (batch, seq_len, d_model) -> (batch, seq_len, d_model)"""
        B_size, T, D = x.shape
        
        # Split into SSM path and gate
        xz = self.in_proj(x)  # (B, T, 2D)
        x_ssm, z = xz.chunk(2, dim=-1)
        
        # Causal conv1d
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        
        # Compute input-dependent SSM parameters
        dt = F.softplus(self.dt_proj(x_conv))  # (B, T, D)
        B_t = self.B_proj(x_conv)               # (B, T, N)
        C_t = self.C_proj(x_conv)               # (B, T, N)
        A = -torch.exp(self.A_log)               # (D, N), negative for stability
        
        # Sequential scan (efficient for T=12)
        y = self._selective_scan(x_conv, dt, A, B_t, C_t)
        
        # Gated output
        y = y * F.silu(z)
        return self.out_proj(y)
    
    def _selective_scan(self, x, dt, A, B, C):
        """Selective scan with input-dependent discretization.
        
        x:  (batch, T, D)
        dt: (batch, T, D) — step sizes
        A:  (D, N) — state matrix (log-space, already negated)
        B:  (batch, T, N) — input matrix
        C:  (batch, T, N) — output matrix
        """
        batch, T, D = x.shape
        N = self.d_state
        
        h = torch.zeros(batch, D, N, device=x.device, dtype=x.dtype)
        ys = []
        
        for t in range(T):
            dt_t = dt[:, t, :, None]          # (batch, D, 1)
            dA = torch.exp(A[None] * dt_t)     # (batch, D, N)
            dB = dt_t * B[:, t, None, :]       # (batch, D, N)
            
            h = h * dA + x[:, t, :, None] * dB  # state update
            y_t = (h * C[:, t, None, :]).sum(-1) # output projection
            ys.append(y_t)
        
        y = torch.stack(ys, dim=1)  # (batch, T, D)
        return y + x * self.D[None, None, :]  # skip connection


class ProtoSSM(nn.Module):
    """Prototypical State Space Model for temporal bioacoustic event detection.
    
    Architecture:
    1. Linear projection of Perch embeddings (1536 -> d_model)
    2. Bidirectional Selective SSM for temporal context
    3. Prototypical cosine similarity classification
    4. Gated fusion with Perch foundation model logits
    
    Args:
        d_input: Perch embedding dimension (1536)
        d_model: Internal model dimension
        d_state: SSM state dimension
        n_ssm_layers: Number of bidirectional SSM layers
        n_classes: Number of species classes (234)
        n_windows: Windows per file (12)
        dropout: Dropout rate
    """
    
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_ssm_layers=2, n_classes=234, n_windows=12, dropout=0.15):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows
        
        # 1. Feature projection
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        # 2. Learnable positional encoding for temporal position within file
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        
        # 3. Bidirectional SSM layers with residual connections
        self.ssm_fwd = nn.ModuleList()
        self.ssm_bwd = nn.ModuleList()
        self.ssm_merge = nn.ModuleList()
        self.ssm_norm = nn.ModuleList()
        for _ in range(n_ssm_layers):
            self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
            self.ssm_norm.append(nn.LayerNorm(d_model))
        self.ssm_drop = nn.Dropout(dropout)
        
        # 4. Learnable class prototypes (initialized from data)
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        
        # 5. Per-class gated fusion with Perch logits
        #    sigmoid(alpha) blends: alpha*proto + (1-alpha)*perch
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
        
        # 6. Taxonomic auxiliary head (set after loading taxonomy)
        self.n_families = 0
        self.family_head = None
    
    def init_prototypes_from_data(self, embeddings, labels):
        """Initialize prototypes as normalized class-mean embeddings.
        
        embeddings: (N, d_input) raw Perch embeddings  
        labels: (N, n_classes) binary label matrix
        """
        with torch.no_grad():
            h = self.input_proj(embeddings)  # (N, d_model)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)
    
    def init_family_head(self, n_families, class_to_family):
        """Initialize taxonomic auxiliary head.
        
        n_families: number of unique families
        class_to_family: (n_classes,) mapping class index to family index
        """
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))
    
    def forward(self, emb, perch_logits=None):
        """
        emb: (B, T, d_input) — Perch embeddings per file
        perch_logits: (B, T, n_classes) — Perch mapped logits (optional)
        
        Returns: 
            species_logits: (B, T, n_classes)
            family_logits: (B, T, n_families) or None
            h_temporal: (B, T, d_model) — for analysis
        """
        B, T, _ = emb.shape
        
        # Project embeddings
        h = self.input_proj(emb)  # (B, T, d_model)
        h = h + self.pos_enc[:, :T, :]
        
        # Bidirectional SSM
        for fwd, bwd, merge, norm in zip(
            self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm
        ):
            residual = h
            h_f = fwd(h)                        # forward scan
            h_b = bwd(h.flip(1)).flip(1)         # backward scan
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)               # residual + layernorm
        
        h_temporal = h  # save for analysis
        
        # Prototypical cosine similarity
        h_norm = F.normalize(h, dim=-1)          # (B, T, d_model)
        p_norm = F.normalize(self.prototypes, dim=-1)  # (C, d_model)
        temp = F.softplus(self.proto_temp)
        sim = torch.matmul(h_norm, p_norm.T) * temp    # (B, T, C)
        
        # Gated fusion with Perch logits
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]  # (1, 1, C)
            species_logits = alpha * sim + (1 - alpha) * perch_logits
        else:
            species_logits = sim
        
        # Taxonomic auxiliary prediction
        family_logits = None
        if self.family_head is not None:
            h_pool = h.mean(dim=1)  # (B, d_model) — file-level
            family_logits = self.family_head(h_pool)  # (B, n_families)
        
        return species_logits, family_logits, h_temporal
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("ProtoSSM architecture defined.")
print(f"Parameter count (d_model=128, 2 layers): {ProtoSSM(d_model=128, n_ssm_layers=2).count_parameters():,}")

In [ ]:
# ProtoSSM Training Loop

def build_family_mapping(taxonomy_df, primary_labels):
    """Build class-to-family index mapping from taxonomy."""
    if "family" not in taxonomy_df.columns:
        # Derive family from class_name or use order as fallback
        if "order" in taxonomy_df.columns:
            family_map = taxonomy_df.set_index("primary_label")["order"].to_dict()
        elif "class_name" in taxonomy_df.columns:
            family_map = taxonomy_df.set_index("primary_label")["class_name"].to_dict()
        else:
            family_map = {label: "Unknown" for label in primary_labels}
    else:
        family_map = taxonomy_df.set_index("primary_label")["family"].to_dict()
    families = sorted(set(family_map.values()))
    fam_to_idx = {f: i for i, f in enumerate(families)}
    class_to_family = []
    for label in primary_labels:
        fam = family_map.get(label, "Unknown")
        class_to_family.append(fam_to_idx.get(fam, 0))
    return len(families), class_to_family, fam_to_idx

def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    """Reshape flat (n_windows*n_files, ...) to (n_files, n_windows, ...).
    
    Groups by filename from meta_df, preserving file order.
    Returns reshaped array and list of unique filenames.
    """
    filenames = meta_df["filename"].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)
    
    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows, \
        f"Expected {n_files * n_windows} rows, got {len(flat_array)}"
    
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files

def train_proto_ssm(model, emb_files, logits_files, labels_files, 
                    file_families=None, cfg=None, verbose=True):
    """Train ProtoSSM with multi-task loss and early stopping.
    
    Args:
        model: ProtoSSM instance
        emb_files: (n_files, n_windows, d_input) Perch embeddings
        logits_files: (n_files, n_windows, n_classes) Perch mapped logits
        labels_files: (n_files, n_windows, n_classes) binary labels
        file_families: (n_files, n_families) multi-hot family labels (optional)
        cfg: training config dict
    
    Returns:
        model with best weights loaded
        training history dict
    """
    if cfg is None:
        cfg = CFG["proto_ssm_train"]
    
    n_files = len(emb_files)
    n_val = max(1, int(n_files * cfg["val_ratio"]))
    
    # Deterministic train/val split by file
    perm = torch.randperm(n_files, generator=torch.Generator().manual_seed(42))
    val_idx = perm[:n_val]
    train_idx = perm[n_val:]
    
    # Convert to tensors
    emb_train = torch.tensor(emb_files[train_idx], dtype=torch.float32)
    logits_train = torch.tensor(logits_files[train_idx], dtype=torch.float32)
    labels_train = torch.tensor(labels_files[train_idx], dtype=torch.float32)
    
    emb_val = torch.tensor(emb_files[val_idx], dtype=torch.float32)
    logits_val = torch.tensor(logits_files[val_idx], dtype=torch.float32)
    labels_val = torch.tensor(labels_files[val_idx], dtype=torch.float32)
    
    # Family labels for auxiliary loss
    fam_train = fam_val = None
    if file_families is not None and model.family_head is not None:
        fam_train = torch.tensor(file_families[train_idx], dtype=torch.float32)
        fam_val = torch.tensor(file_families[val_idx], dtype=torch.float32)
    
    # Class weights for imbalanced data
    pos_counts = labels_train.sum(dim=(0, 1))  # (C,)
    total = labels_train.shape[0] * labels_train.shape[1]
    pos_weight = ((total - pos_counts) / (pos_counts + 1)).clamp(max=cfg["pos_weight_cap"])
    
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg["lr"], 
        epochs=cfg["n_epochs"], steps_per_epoch=1,
        pct_start=0.1, anneal_strategy='cos'
    )
    
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    
    for epoch in range(cfg["n_epochs"]):
        # === Train ===
        model.train()
        species_out, family_out, _ = model(emb_train, logits_train)
        
        # Primary loss: weighted BCE
        loss_bce = F.binary_cross_entropy_with_logits(
            species_out, labels_train,
            pos_weight=pos_weight[None, None, :]
        )
        
        # Knowledge distillation loss: MSE between model output and Perch logits
        loss_distill = F.mse_loss(species_out, logits_train)
        
        # Total loss
        loss = loss_bce + cfg["distill_weight"] * loss_distill
        
        # Taxonomic auxiliary loss
        if family_out is not None and fam_train is not None:
            loss_family = F.binary_cross_entropy_with_logits(family_out, fam_train)
            loss = loss + 0.1 * loss_family
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        # === Validate ===
        model.eval()
        with torch.no_grad():
            val_out, val_fam, _ = model(emb_val, logits_val)
            val_loss = F.binary_cross_entropy_with_logits(
                val_out, labels_val,
                pos_weight=pos_weight[None, None, :]
            )
            
            # Compute validation AUC
            val_pred = val_out.reshape(-1, val_out.shape[-1]).numpy()
            val_true = labels_val.reshape(-1, labels_val.shape[-1]).numpy()
            try:
                val_auc = macro_auc_skip_empty(val_true, val_pred)
            except Exception:
                val_auc = 0.0
        
        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss.item())
        history["val_auc"].append(val_auc)
        
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        
        if verbose and (epoch + 1) % 20 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
                  f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}")
        
        if wait >= cfg["patience"]:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})")
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    if verbose:
        print(f"  Training complete. Best val_loss={best_val_loss:.4f}")
        # Report fusion alpha distribution
        with torch.no_grad():
            alphas = torch.sigmoid(model.fusion_alpha).numpy()
            print(f"  Fusion alpha: mean={alphas.mean():.3f} min={alphas.min():.3f} max={alphas.max():.3f}")
            print(f"  Proto temperature: {F.softplus(model.proto_temp).item():.3f}")
    
    return model, history

print("ProtoSSM training functions defined.")

In [ ]:
# Cell 15 — Pseudo-labeling loop

def reshape_to_files(arr, meta_df):
    """Reshape flat (n_windows_total, D) to (n_files, N_WINDOWS, D)."""
    fnames = meta_df["filename"].values
    unique_files = list(dict.fromkeys(fnames))  # preserve order
    n_files = len(unique_files)
    D = arr.shape[1] if arr.ndim == 2 else arr.shape[1:]

    if arr.ndim == 2:
        out = np.zeros((n_files, N_WINDOWS, arr.shape[1]), dtype=arr.dtype)
    else:
        out = np.zeros((n_files, N_WINDOWS) + arr.shape[1:], dtype=arr.dtype)

    file_to_idx = {f: i for i, f in enumerate(unique_files)}
    window_counters = np.zeros(n_files, dtype=int)
    for row_idx, fn in enumerate(fnames):
        fi = file_to_idx[fn]
        wi = window_counters[fi]
        if wi < N_WINDOWS:
            out[fi, wi] = arr[row_idx]
            window_counters[fi] += 1

    return out, unique_files

def train_full_pipeline(emb_data, scores_data, Y_data, meta_data, cfg, round_name=""):
    """Train ProtoSSM + MLP probes on given data. Returns model, probes, and fitted transformers."""
    print(f"\n{'='*60}")
    print(f"Training pipeline: {round_name}")
    print(f"Data: {len(meta_data)} windows, {meta_data['filename'].nunique()} files")
    print(f"{'='*60}")

    # Binarize labels for classifiers (pseudo-labels are soft floats)
    Y_binary = (Y_data >= 0.5).astype(np.float32)

    # Fit PCA on embeddings
    emb_scaler = StandardScaler()
    emb_scaled = emb_scaler.fit_transform(emb_data)
    emb_pca = PCA(n_components=min(cfg["pca_dim"], emb_data.shape[1]), random_state=42)
    Z = emb_pca.fit_transform(emb_scaled).astype(np.float32)

    # Reshape to file-level
    emb_files, file_list = reshape_to_files(emb_data, meta_data)
    logits_files, _ = reshape_to_files(scores_data, meta_data)
    labels_files, _ = reshape_to_files(Y_data, meta_data)

    # Build family mapping
    n_families, class_to_family, fam_to_idx = build_family_mapping(taxonomy, PRIMARY_LABELS)
    file_families = np.zeros((len(file_list), n_families), dtype=np.float32)
    for fi in range(len(file_list)):
        active_classes = np.where(labels_files[fi].sum(axis=0) > 0)[0]
        for ci in active_classes:
            file_families[fi, class_to_family[ci]] = 1.0

    # ProtoSSM
    ssm_cfg = cfg["proto_ssm"]
    model = ProtoSSM(
        d_input=emb_data.shape[1],
        d_model=ssm_cfg["d_model"],
        d_state=ssm_cfg["d_state"],
        n_ssm_layers=ssm_cfg["n_ssm_layers"],
        n_classes=N_CLASSES,
        n_windows=N_WINDOWS,
        dropout=ssm_cfg["dropout"],
    ).to(DEVICE)

    emb_flat_tensor = torch.tensor(emb_data, dtype=torch.float32)
    labels_flat_tensor = torch.tensor(Y_data, dtype=torch.float32)
    model.init_prototypes_from_data(emb_flat_tensor, labels_flat_tensor)
    model.init_family_head(n_families, class_to_family)

    t0 = time.time()
    model, history = train_proto_ssm(
        model,
        emb_files, logits_files, labels_files.astype(np.float32),
        file_families=file_families,
        cfg=cfg["proto_ssm_train"],
        verbose=True,
    )
    print(f"ProtoSSM training: {time.time()-t0:.1f}s")

    # OOF base/prior for MLP features
    sc_meta = meta_data.copy()
    groups = sc_meta["filename"].to_numpy()

    # Fit prior tables
    prior_tables = fit_prior_tables(sc_meta.reset_index(drop=True), Y_binary)

    # Build base scores with prior fusion
    base_scores, prior_scores = fuse_scores_with_tables(
        scores_data,
        sites=sc_meta["site"].to_numpy(),
        hours=sc_meta["hour_utc"].to_numpy(),
        tables=prior_tables,
    )

    # Train MLP probes (using binarized labels)
    min_pos = int(cfg["frozen_best_probe"]["min_pos"])
    PROBE_CLASS_IDX = np.where(Y_binary.sum(axis=0) >= min_pos)[0].astype(np.int32)

    probe_models = {}
    for cls_idx in tqdm(PROBE_CLASS_IDX, desc="MLP probes", disable=not cfg["verbose"]):
        y = Y_binary[:, cls_idx]
        if y.sum() == 0 or y.sum() == len(y):
            continue

        X_cls = build_class_features(
            Z,
            raw_col=scores_data[:, cls_idx],
            prior_col=prior_scores[:, cls_idx],
            base_col=base_scores[:, cls_idx],
        )

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos
        if n_pos > 0 and n_neg > n_pos:
            repeat = max(1, n_neg // n_pos)
            pos_idx = np.where(y == 1)[0]
            X_bal = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
            y_bal = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
        else:
            X_bal, y_bal = X_cls, y

        clf = MLPClassifier(**cfg["mlp_params"])
        clf.fit(X_bal, y_bal)
        probe_models[cls_idx] = clf

    print(f"MLP probes trained: {len(probe_models)}")

    return {
        "model": model,
        "probes": probe_models,
        "emb_scaler": emb_scaler,
        "emb_pca": emb_pca,
        "prior_tables": prior_tables,
    }

def predict_scores(pipeline, emb_data, scores_data, meta_data):
    """Run full prediction pipeline (ProtoSSM + MLP + ensemble)."""
    model = pipeline["model"]
    probes = pipeline["probes"]
    emb_scaler = pipeline["emb_scaler"]
    emb_pca = pipeline["emb_pca"]
    prior_tables = pipeline["prior_tables"]

    # ProtoSSM inference
    emb_files, file_list = reshape_to_files(emb_data, meta_data)
    logits_files, _ = reshape_to_files(scores_data, meta_data)

    model.eval()
    with torch.no_grad():
        proto_out, _, _ = model(
            torch.tensor(emb_files, dtype=torch.float32),
            torch.tensor(logits_files, dtype=torch.float32),
        )
        proto_scores_flat = proto_out.numpy().reshape(-1, N_CLASSES).astype(np.float32)

    # Prior fusion
    base_scores, prior_scores = fuse_scores_with_tables(
        scores_data,
        sites=meta_data["site"].to_numpy(),
        hours=meta_data["hour_utc"].to_numpy(),
        tables=prior_tables,
    )

    # MLP probe scores
    emb_scaled = emb_scaler.transform(emb_data)
    Z = emb_pca.transform(emb_scaled).astype(np.float32)

    mlp_scores = base_scores.copy()
    alpha = float(CFG["frozen_best_probe"]["alpha"])
    for cls_idx, clf in probes.items():
        X_cls = build_class_features(
            Z,
            raw_col=scores_data[:, cls_idx],
            prior_col=prior_scores[:, cls_idx],
            base_col=base_scores[:, cls_idx],
        )
        if hasattr(clf, "predict_proba"):
            prob = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
            pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        else:
            pred = clf.decision_function(X_cls).astype(np.float32)
        mlp_scores[:, cls_idx] = (1.0 - alpha) * base_scores[:, cls_idx] + alpha * pred

    # Ensemble
    final_scores = 0.5 * proto_scores_flat + 0.5 * mlp_scores
    return final_scores.astype(np.float32)

print("Pseudo-labeling functions ready.")

In [ ]:
# Cell 16 — Run pseudo-labeling rounds

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# Round 0: Train on labeled data only
pipeline = train_full_pipeline(
    emb_labeled, scores_labeled, Y_labeled, meta_labeled,
    cfg=CFG, round_name="Round 0 (labeled only)"
)

for round_idx in range(CFG["pseudo_rounds"]):
    power = CFG["pseudo_power"][round_idx] if round_idx < len(CFG["pseudo_power"]) else CFG["pseudo_power"][-1]
    print(f"\n{'#'*60}")
    print(f"# PSEUDO-LABELING ROUND {round_idx + 1} (power={power:.2f})")
    print(f"{'#'*60}")

    # 1. Predict on unlabeled data
    t0 = time.time()
    unlabeled_scores = predict_scores(
        pipeline, emb_unlabeled, scores_unlabeled, meta_unlabeled
    )
    print(f"Prediction on {len(meta_unlabeled)} unlabeled windows: {time.time()-t0:.1f}s")

    # 2. Convert to probabilities and apply Power Transform
    unlabeled_probs = sigmoid(unlabeled_scores)
    unlabeled_pseudo = unlabeled_probs ** power
    print(f"Power transform (power={power:.2f}): max={unlabeled_pseudo.max():.3f}, "
          f"mean={unlabeled_pseudo.mean():.5f}")

    # 3. Create pseudo-labels — filter at FILE level (must keep all 12 windows per file)
    threshold = CFG["pseudo_threshold"]
    file_max_prob = meta_unlabeled.groupby("filename").apply(
        lambda g: unlabeled_pseudo[g.index].max() > threshold
    )
    mask = meta_unlabeled["filename"].map(file_max_prob).values
    n_pseudo_windows = mask.sum()
    n_pseudo_files = file_max_prob.sum()
    print(f"Pseudo-labeled files: {n_pseudo_files}/{meta_unlabeled['filename'].nunique()} "
          f"({n_pseudo_windows} windows, threshold={threshold})")

    if n_pseudo_files == 0:
        print("No pseudo-labels generated. Stopping.")
        break

    # 4. Combine labeled + pseudo-labeled data
    pseudo_meta = meta_unlabeled[mask].reset_index(drop=True)
    pseudo_emb = emb_unlabeled[mask]
    pseudo_scores = scores_unlabeled[mask]
    pseudo_Y = unlabeled_pseudo[mask]  # soft pseudo-labels

    combined_meta = pd.concat([meta_labeled, pseudo_meta], ignore_index=True)
    combined_emb = np.vstack([emb_labeled, pseudo_emb])
    combined_scores = np.vstack([scores_labeled, pseudo_scores])
    combined_Y = np.vstack([Y_labeled, pseudo_Y])

    print(f"Combined data: {len(combined_meta)} windows "
          f"({len(meta_labeled)} labeled + {n_pseudo_windows} pseudo)")

    # 5. Retrain
    pipeline = train_full_pipeline(
        combined_emb, combined_scores, combined_Y, combined_meta,
        cfg=CFG, round_name=f"Round {round_idx + 1} (labeled + pseudo)"
    )

    # 6. Evaluate on labeled data (sanity check)
    labeled_preds = predict_scores(
        pipeline, emb_labeled, scores_labeled, meta_labeled
    )
    labeled_probs = sigmoid(labeled_preds)
    auc = macro_auc_skip_empty(Y_labeled, labeled_probs)
    print(f"\nLabeled data AUC after round {round_idx + 1}: {auc:.4f}")

print("\nPseudo-labeling complete.")

In [ ]:
# Cell 17 — Save trained weights and models

# Save ProtoSSM weights
proto_path = OUT_DIR / "pseudo_label_proto_ssm.pt"
torch.save({
    "model_state_dict": pipeline["model"].state_dict(),
    "config": CFG["proto_ssm"],
    "n_classes": N_CLASSES,
    "n_windows": N_WINDOWS,
}, proto_path)
print(f"Saved ProtoSSM: {proto_path}")

# Save MLP probes
probe_path = OUT_DIR / "pseudo_label_probes.pkl"
with open(probe_path, "wb") as f:
    pickle.dump(pipeline["probes"], f)
print(f"Saved MLP probes: {probe_path}")

# Save transformers (scaler, PCA, prior tables)
aux_path = OUT_DIR / "pseudo_label_aux.pkl"
with open(aux_path, "wb") as f:
    pickle.dump({
        "emb_scaler": pipeline["emb_scaler"],
        "emb_pca": pipeline["emb_pca"],
        "prior_tables": pipeline["prior_tables"],
        "cfg": CFG,
    }, f)
print(f"Saved auxiliary: {aux_path}")

# Save embeddings cache (for future use)
print(f"\nEmbeddings cache: {CACHE_PATH}")
print(f"\nAll output files:")
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1e6:.1f} MB")

print("\nDone! Download output files and upload as Kaggle Dataset for submission notebook.")